# Build and Evaluate Custom Metrics with WatsonX.AI LLMaaJ Provider for PTA using One step configuration


This notebook should be run using Runtime 24.1 & Python 3.11 or greater runtime environment. if you are viewing this in Watson Studio and do not see Python 3.11.x in the upper right corner of your screen, please update the runtime now.

This notebook demonstrates how to evaluate custom metrics for prompt templates using the LLM as a Judge (LLMaaJ) evaluator, enabling domain-specific evaluation beyond built-in generative AI quality metrics, where a large language model acts as a judge to assess AI-generated responses based on customizable grading logic.

The notebook will create a summarization prompt template asset in a given deployment space, configure OpenScale to monitor that PTA and evaluate custom metrics using WatsonX.AI llmaaj evaluator and model health metrics.

If users wish to execute this notebook for task types other than summarization, please consult [this](https://github.com/IBM/watson-openscale-samples/blob/main/IBM%20Cloud/WML/notebooks/watsonx/README.md) document for guidance on evaluating prompt templates for the available task types.

**Note:** User can search for `EDIT THIS` and fill the inputs needed.

## Contents

- [Package installation and dependencies](#package-installation-and-dependencies)
- [Credentials setup](#credentials-setup)
- [Sample data loading and preview](#sample-data-loading-and-preview)
- [Create prompt template asset](#create-prompt-template-asset)
- [Create deployment](#create-deployment)
- [Create openscale client](#create-openscale-client)
- [Setup LLM as a judge configuration](#setup-llm-as-a-judge-configuration)
    - [Build the WatsonX.AI LLM Provider Configuration](#build-the-watsonxai-llm-provider-configuration)
    - [Define Custom Metrics](#define-custom-metrics)
    - [Set Up the Custom Monitor (One-Step Configuration)](#set-up-the-custom-monitor-one-step-configuration)
- [Upload records for production space evaluation](#upload-records-for-production-space-evaluation)
- [Risk evaluation](#risk-evaluation)
- [View results](#view-results)
    - [Display metric scores](#display-metric-scores)
    - [Display record-level metrics](#display-record-level-metrics)

## Prerequisite

* It requires service credentials for IBM Watson OpenScale:
* Requires a CSV file containing the test data that needs to be evaluated
* Requires the ID of deployment space in which you want to create the prompt template asset.

# Package installation and dependencies

Install all necessary Python packages for this notebook.
- `ibm-aigov-facts-client`: For factsheets
- `ibm-watson-openscale`: IBM OpenScale SDK
  
**Restart the kernel after installation before continuing.**

In [ ]:
!pip install --upgrade ibm-aigov-facts-client | tail -n 1
!pip install --upgrade ibm-watson-openscale | tail -n 1

# Credentials setup

Configure authentication credentials and environment settings:
- `CPD_URL`: Cloud Pak for Data cluster URL
- `CPD_USERNAME`, `CPD_PASSWORD`: Authentication credentials
- `CPD_API_KEY`: Optional API key for authentication (alternative to username/password)
- `CPD_VERSION`: Cloud Pak for Data version (e.g., "5.3")
- `SERVICE_INSTANCE_ID`: Watson OpenScale datamart identifier
- `SPACE_ID`: Watson Studio deployment space ID where prompt template will be created
- `PROVIDER_TYPE`: Judge model provider type
- `MODEL_ID`: Specific judge model identifier for LLM-based evaluation
- `CUSTOM_MONITOR_NAME`: Unique identifier for the custom monitor

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
# ── Cloud Pak for Data credentials ─────────────────────────────────────────────
CPD_URL      = "<EDIT THIS>"           # e.g. https://cpd-cpd-instance.apps.example.com
CPD_USERNAME = "cpadmin"               # e.g. "cpadmin"
CPD_PASSWORD = "<EDIT THIS>"
# CPD_API_KEY  = "<EDIT THIS>"         # Optional: If using API key instead of username/password
CPD_VERSION  = "5.3"
SERVICE_INSTANCE_ID = "00000000-0000-0000-0000-000000000000"    # e.g. "00000000-0000-0000-0000-000000000000"

In [5]:
# ── Project and judge model settings ─────────────────────────────────────────
SPACE_ID = "<EDIT THIS>"               # Watson Studio project ID
PROVIDER_TYPE = "watsonx.ai"           # Type of judge model provider
MODEL_ID   = "ibm/granite-3-3-8b-instruct" # Judge model used for LLM-based evaluation
WML_LOCATION = "cpd_local"             # Supported wml locations are cpd_local, cpd_remote and cloud_remote

In [7]:
# Enter CUSTOM MONITOR NAME
CUSTOM_MONITOR_NAME = "summarization_quality" 

### Create the access token

Generate a temporary access token from CPD credentials for API authentication. The token expires after a period of time; re-run this cell if authentication errors occur.

In [8]:
import requests
import json

def generate_access_token():
    """Generate a CPD access token from username/password."""
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json"
    }
    data = json.dumps({"username": CPD_USERNAME, "password": CPD_PASSWORD}).encode("utf-8")
    response = requests.post(
        CPD_URL + "/icp4d-api/v1/authorize",
        data=data, headers=headers, verify=False
    )
    return response.json()["token"]

iam_access_token = generate_access_token()
print("✅ Access token generated.")

✅ Access token generated.


# Sample data loading and preview

Download or read the CSV file containing test data

**Note:** The `generated_text` column must contain pre-generated outputs from your model


In [9]:
# Download sample Summarization test data.
# Replace this URL with your own data source if needed.
!rm -fr llm_content.csv
!wget "https://raw.githubusercontent.com/IBM/watson-openscale-samples/main/IBM%20Cloud/WML/assets/data/watsonx/llm_content.csv"

--2026-03-26 15:30:58--  https://raw.githubusercontent.com/IBM/watson-openscale-samples/main/IBM%20Cloud/WML/assets/data/watsonx/llm_content.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 42771 (42K) [application/octet-stream]
Saving to: ‘llm_content.csv’

llm_content.csv     100%[===================>]  41.77K  --.-KB/s    in 0.007s  

2026-03-26 15:30:58 (6.11 MB/s) - ‘llm_content.csv’ saved [42771/42771]



In [10]:
import pandas as pd

TEST_DATA_PATH = "llm_content.csv"
label_column = "reference_summary_1"

# Load the data
llm_data = pd.read_csv(TEST_DATA_PATH, encoding="utf-8", encoding_errors="ignore")

# Rename column from generated_summary to generated_text
llm_data = llm_data.rename(columns={'generated_summary': 'generated_text'})

# Optional: limit to first N rows for a quick test run
llm_data = llm_data.head(10)
llm_data.to_csv(TEST_DATA_PATH, index=False)

print(f"✅ Loaded {len(llm_data)} rows. Columns: {list(llm_data.columns)}")
llm_data

✅ Loaded 10 rows. Columns: ['input_text', 'generated_text', 'reference_summary_1', 'reference_summary_2']


,input_text,generated_text,reference_summary_1,reference_summary_2
0,Scientists have discovered a new species of de...,New bioluminescent fish species found in deep ...,Discovery of deep-sea fish emitting soothing l...,Scientists find new bioluminescent fish specie...
1,An international team of astronomers has ident...,Distant exoplanet\'s water vapor-filled atmosp...,Astronomers identify exoplanet with water vapo...,Discovery of exoplanet with water vapor in its...
2,Researchers have developed a novel nanotechnol...,New nanotechnology-based cancer treatment demo...,Researchers create cancer treatment using nano...,Innovative cancer treatment utilizing nanotech...
3,A new app is aiming to reduce food waste by co...,App connects local restaurants with customers ...,New sustainability-focused app facilitates sal...,Initiative to reduce food waste involves app c...
4,Archaeologists have uncovered an ancient city ...,"Ancient city dating back over 4,000 years disc...",Archaeological find in Iraq reveals ancient ci...,"Discovery of 4,000-year-old ancient city in Ku..."
5,Researchers have developed a new type of solar...,Scientists create highly efficient solar panel...,New solar panels could revolutionize renewable...,Researchers develop innovative solar panels to...
6,"According to a recent study, regular exercise ...",Middle-age exercise linked to 36% lower risk o...,New study suggests that staying physically act...,Regular exercise during middle age may have a ...
7,Scientists have developed a new type of batter...,New wireless charging battery utilizes radio w...,Battery technology innovation enables wireless...,Researchers create battery that can be charged...
8,A team of engineers has designed a lightweight...,Engineers create lightweight exoskeleton to ai...,Newly designed exoskeleton offers mobility ass...,Innovative exoskeleton design aims to assist p...
9,NASA\'s latest space telescope has captured st...,New NASA space telescope images unveil distant...,Stunning images of distant galaxies captured b...,Images from NASA\'s latest space telescope off...


# Create prompt template asset

Create a prompt template asset in the Watson Studio deployment space for the summarization task. The template includes:
- `task_id`: Task type identifier ("summarization")
- `model_id`: Model used for generating summaries
- `prompt_input`: Template text with variable placeholders (e.g., `{input_text}`)
- `prompt_variables`: Dictionary defining input variables
- `input_prefix` / `output_prefix`: Optional text added before/after the prompt

Returns the asset ID for use in OpenScale configuration.

Initialize the AI Governance Facts Client to manage prompt template assets in the Watson Studio deployment space. Configuration includes:
- `service_url`: CPD cluster URL
- `username` / `password`: Authentication credentials
- `container_id`: deployment space ID where assets will be stored
- `container_type`: Container type ("space" for Watson Studio deployment spaces)
- `disable_tracing`: Disables telemetry tracing

In [11]:
from ibm_aigov_facts_client import AIGovFactsClient, CloudPakforDataConfig

creds = CloudPakforDataConfig(
    service_url=CPD_URL, username=CPD_USERNAME, password=CPD_PASSWORD
)
facts_client = AIGovFactsClient(
    cloud_pak_for_data_configs=creds,
    container_id=SPACE_ID,
    container_type="space",
    disable_tracing=True,
)

In [ ]:
from ibm_aigov_facts_client import PromptTemplate

task_id = "summarization"
name = "PTA for summarization with LLMaaJ Custom Metrics using WatsonX.AI"
description = "PTA for summarization with LLMaaJ Custom Metrics using WatsonX.AI sample notebook"
model_id = MODEL_ID
prompt_input = "Summarize the following text:\n\n{input_text}\n\nSummary:"

prompt_variables = {"input_text": {}}
input = prompt_input
input_prefix = ""
output_prefix = ""

prompt_template = PromptTemplate(
    input=input,
    prompt_variables=prompt_variables,
    input_prefix=input_prefix,
    output_prefix=output_prefix,
)

pta_details = facts_client.assets.create_prompt(
    input_mode = "structured",
    model_id=model_id,
    task_id=task_id,
    name=name,
    description=description,
    prompt_details=prompt_template
)

space_pta_id = pta_details.to_dict()["asset_id"]
space_pta_id

2026/03/26 15:31:04 INFO : ------------------------------ Structured Prompt Creation Started ------------------------------
2026/03/26 15:31:07 INFO : The structured prompt with ID b199df20-b88d-4846-989a-1abadd7ff54f was created successfully in container_id cbcbfcbf-e6ad-4fbf-ad0f-673cd5ba10cc.


'b199df20-b88d-4846-989a-1abadd7ff54f'

## Create deployment
To create a subscription from a space, it is necessary to create a deployment for a PTA in a space.

In [13]:
from datetime import datetime

DEPLOYMENTS_URL = CPD_URL + "/ml/v4/deployments"

headers = {}
headers["Content-Type"] = "application/json"
headers["Accept"] = "*/*"
headers["Authorization"] = "Bearer {}".format(iam_access_token)

payload = {
    "prompt_template": {"id": space_pta_id},
    "online": {
    },
    "base_model_id": MODEL_ID,
    "description": "rag qa deployment",
    "name": "LLMaaJ as custom_monitor Testing",  # PTA asset name
    "space_id": SPACE_ID
}

version = datetime.now().strftime("%Y-%m-%d")  # The version date for the API of the form YYYY-MM-DD. Example : 2023-07-07
params = {"version": version}

response = requests.post(
    DEPLOYMENTS_URL, json=payload, headers=headers, params=params, verify=False
)
json_data = response.json()

if "metadata" in json_data:
    deployment_id = json_data["metadata"]["id"]
    print("Deployment ID: ", deployment_id)
else:
    print(json_data)

Deployment ID:  72055147-d23c-4b62-95a2-93b7984c5f46


# Create openscale client

Initialize the Watson OpenScale client for model monitoring and governance.

In [14]:
from ibm_cloud_sdk_core.authenticators import CloudPakForDataAuthenticator
from ibm_watson_openscale import APIClient

authenticator = CloudPakForDataAuthenticator(
    url=CPD_URL,
    username=CPD_USERNAME,
    password=CPD_PASSWORD,
    disable_ssl_verification=True
)
wos_client = APIClient(
    service_url=CPD_URL,
    authenticator=authenticator,
    service_instance_id=SERVICE_INSTANCE_ID
)

print(f"✅ Watson OpenScale client initialized. SDK version: {wos_client.version}")

✅ Watson OpenScale client initialized. SDK version: 1.0.13


## Map openscale instance to deployment space

When the authentication is on CPD then we need to add additional step of mapping the `space_id` to an OpenScale instance.

In [15]:
from ibm_watson_openscale.base_classes import ApiRequestFailure

try:
    wos_client.wos.add_instance_mapping(
        service_instance_id=SERVICE_INSTANCE_ID, space_id=SPACE_ID
    )
except ApiRequestFailure as arf:
    if arf.response.status_code == 409:
        # Instance mapping already exists
        pass
    else:
        raise arf

Instance mapping already exists. Skipping creation.


# Setup LLM as a judge configuration

### Build the WatsonX.AI LLM Provider Configuration

Configure the judge model that evaluates RAG outputs. The configuration specifies:
- `username`, `password`: CPD authentication credentials
- `apikey`: Optional API key for authentication
- `space_id`: Watson Studio space ID
- `url`: CPD cluster URL
- `type`: Judge model provider type (e.g., "watsonx.ai")
- `model_id`: Specific judge model identifier
- `wml_location`: "cpd_local" for on-premises CPD deployment

Note: For supported evaluator types, refer to: https://github.com/IBM/watson-openscale-samples/wiki/Generative-AI-Evaluator-templates#supported-evaluator-types

In [16]:
# Configuration for the judge model (the LLM that evaluates your RAG outputs)
LLM_PROVIDER_CONFIG = {
    "username":    CPD_USERNAME,
    "password":    CPD_PASSWORD,
    # "apikey":      CPD_API_KEY,       # uncomment this if using api key for authentication
    "space_id":    SPACE_ID,
    "url":         CPD_URL,
    "type":        PROVIDER_TYPE,
    "model_id":    MODEL_ID,
    "wml_location": WML_LOCATION,
}

print("✅ LLM provider config ready.")

✅ LLM provider config ready.


### Define Custom Metrics
**prompt:**
- Defines the instruction provided to the LLM judge describing how a response should be evaluated.  
- The prompt contains placeholders that will later be replaced with values from the evaluation dataset during metric computation.

**grading_options:**
- Defines the scoring scale used by the LLM judge. 
- Each option represents a possible evaluation outcome and is associated with a numeric score that will be recorded as the metric result.

**grader_prompt_variables_mapping:**
- Maps placeholders used inside the evaluation prompt to the corresponding column names in the dataset.  
- This mapping ensures that the correct dataset values are substituted into the prompt before it is sent to the LLM judge for evaluation.

Each metric configuration includes:
- `name`: Metric identifier
- `description`: Explanation of what the metric evaluates
- `thresholds`: Acceptable score ranges (`upper_limit`, `lower_limit`)
- `computation`: Evaluation logic
  - `prompt`: Grading instructions for the judge LLM with variable placeholders
  - `grading_options`: Possible evaluation outcomes with assigned scores
    - `name`: Option identifier
    - `description`: What this option represents
    - `value`: Numeric score (0-1 scale)
- `grader_prompt_variables_mapping`: Maps prompt variables to CSV columns
- `thresholds`: Acceptable value ranges
  - `upper_limit`: Maximum acceptable value
  - `lower_limit`: Minimum acceptable value
- `dataset_type`: Data source for evaluation
  - `payload_logging`: Uses model-generated output (`generated_text`)
  - `feedback`: Uses reference data (`reference_summary_1`)

In [17]:
# ── Custom Metrics ────────────────────────────────────────────────────
# Each metric uses an LLM grader prompt to score a quality dimension.

MONITOR_METRICS = [

    # ── Metric 1: Coherence ────────────────────────────────────────
    {
        "name": "coherence",
        "description": "Evaluates whether the generated summary forms a coherent and meaningful summary of the source text.",
        "computation": {
            "prompt": (
                "You are an EXTREMELY STRICT evaluator.\n\n"

                "Source Text:\n{source}\n\n"
                "Generated Summary:\n{summary}\n\n"

                "Your job is to FIND FAULTS. Assume the summary is NOT perfect.\n\n"

                "Carefully check:\n"
                "- Is ANY important detail missing? (numbers, cause, outcome)\n"
                "- Is the summary slightly vague or generic?\n"
                "- Is any nuance from the source lost?\n\n"

                "STRICT RULES:\n"
                "- If ANY detail is missing → NOT perfect\n"
                "- If slightly generic → NOT perfect\n"
                "- Only give 'coherent' if it is COMPLETE and PRECISE\n\n"

                "Label rules:\n"
                "coherent → PERFECT (no missing detail, no vagueness)\n"
                "partial → ANY imperfection at all\n"
                "incoherent → wrong or misleading\n\n"

                "Most summaries are NOT perfect.\n"
                "If unsure, DO NOT choose coherent.\n\n"

                "Return ONLY one label:\n"
                "coherent\n"
                "partial\n"
                "incoherent\n\n"

                "Do NOT explain.\n"
                "Do NOT add punctuation.\n"
                "Output must be EXACT."
            ),
            "grading_options": [
                {"name": "coherent", "value": 1},
                {"name": "partial", "value": 0.5},
                {"name": "incoherent", "value": 0}
            ]
        },
        "grader_prompt_variables_mapping": {
            "source": "input_text",
            "summary": "generated_text"
        },
        "dataset_type": "payload_logging",
        "thresholds": {
            "upper_limit": 0.8,
            "lower_limit": 0.2
        }
    },

    # ── Metric 2: Fluency ────────────────────────────────────────
    {
        "name": "fluency",
        "description": "Evaluates grammatical correctness and readability of the generated summary.",
        "computation": {
            "prompt": (
                "You are an EXTREMELY STRICT grammar evaluator.\n\n"

                "Summary:\n{summary}\n\n"

                "Your job is to FIND EVEN THE SMALLEST flaw.\n\n"

                "Check carefully:\n"
                "- Is phrasing slightly unnatural?\n"
                "- Is the sentence slightly compressed or robotic?\n"
                "- Is flow not perfectly smooth?\n"
                "- Would a native speaker improve it slightly?\n\n"

                "STRICT RULES:\n"
                "- If ANY minor issue exists → NOT perfect\n"
                "- Only perfectly natural, human-like sentences get 'fluent'\n\n"

                "Label rules:\n"
                "fluent → PERFECT (native-level natural, zero issues)\n"
                "partial → ANY small issue at all\n"
                "disfluent → clear grammar problems\n\n"

                "Most summaries are NOT perfect.\n"
                "If unsure, DO NOT choose fluent.\n\n"

                "Return ONLY one label:\n"
                "fluent\n"
                "partial\n"
                "disfluent\n\n"

                "Do NOT explain.\n"
                "Do NOT add punctuation.\n"
                "Output must be EXACT."
            ),
            "grading_options": [
                {"name": "fluent", "value": 1},
                {"name": "partial", "value": 0.5},
                {"name": "disfluent", "value": 0}
            ]
        },
        "grader_prompt_variables_mapping": {
            "summary": "reference_summary_1"
        },
        "dataset_type": "feedback",
        "thresholds": {
            "upper_limit": 0.7,
            "lower_limit": 0.1
        }
    }
]

#### LLM Judge Configuration

Complete configuration for LLM-as-a-Judge custom monitoring:
- **DATAMART_ID**: Watson OpenScale service instance ID
- **LLM_PROVIDER_CONFIG**: Judge model configuration (defined above)
- **PROMPT_TEMPLATE_ASSET_ID**: Created prompt template asset ID
- **SPACE_ID**: Watson Studio deployment space ID
- **DEPLOYMENT_ID**: Deployment id from deployed pta asset
- **operational_space_id:** Set to `"pre_production"` or `"production"`.
  - Use `"pre_production"` for staging or testing environments.
  - Use `"production"` for live production monitoring.
- **PROBLEM_TYPE**: Specifies the problem type being monitored. e.g: "retrieval_augmented_generation"
- **MIN_RECORDS**, **MAX_RECORDS**: Evaluation record limits
- **CUSTOM_MONITOR_CONFIG**: Custom monitor settings
  - `CUSTOM_MONITOR_NAME`: Monitor definition identifier
  - `MONITOR_METRICS`: List of custom metrics (defined above)
- **DELETE_LLM_PROVIDER**: Set to `False` if don't want to delete llm provider, default value is `True`
- **DELETE_CUSTOM_MONITOR**: Set to `False` if don't want to delete custom monitor, default value is `True`

**Note:** Default schedule interval is 1 hour, to modify schedule interval, set `ENABLE_SCHEDULE=True` and pass schedule payload under `CUSTOM_MONITOR_CONFIG`

In [18]:
# Full LLMaaJ configuration dictionary
llmaj_config = {

    # ── Watson OpenScale data mart ────────────────────────────────────────────
    "DATAMART_ID": SERVICE_INSTANCE_ID,

    # ── Judge model ───────────────────────────────────────────────────────────
    "LLM_PROVIDER_CONFIG": LLM_PROVIDER_CONFIG,

    # ── Prompt template asset ─────────────────────────────────────────────────
    "PROMPT_TEMPLATE_ASSET_ID": space_pta_id,
    "SPACE_ID":                 SPACE_ID,
    "DEPLOYMENT_ID":            deployment_id,         

    # ── Subscription settings ─────────────────────────────────────────────────
    "LABEL_COLUMN":         "reference_summary_1",      # Reference summary column
    "OPERATIONAL_SPACE_ID": "production",               # update to pre_production if using pre production deployment space   
    "PROBLEM_TYPE":         "summarization",            # Update if using different task type

    # ── Summarization-specific field mappings ─────────────────────────────────
    "INPUT_FIELD":  "input_text",

    # ── Evaluation record limits ──────────────────────────────────────────────
    "MIN_RECORDS": 10,

    # ── Custom monitor definition ─────────────────────────────────────────────
    "CUSTOM_MONITOR_CONFIG": {
        "CUSTOM_MONITOR_NAME": CUSTOM_MONITOR_NAME,
        "MONITOR_METRICS":     MONITOR_METRICS
    },

    # ── Cleanup options ───────────────────────────────────────────────────────
    # Set to True to delete and recreate the monitor if it already exists
    "DELETE_LLM_PROVIDER":  True,
    "DELETE_CUSTOM_MONITOR": True
}

## Set Up the Custom Monitor (One-Step Configuration)

This single SDK call automates the entire setup process:

1. Creates/Reuses LLM Provider
2. Creates Custom Monitor Definition
3. Creates Subscription
4. Creates Monitor Instance
5. Creates Custom Dataset

**Note:** If you run this cell multiple times, existing resources will be deleted and recreated (controlled by `DELETE_CUSTOM_MONITOR` and `DELETE_LLM_PROVIDER` flags in the config).

In [19]:
result = wos_client.custom_monitor.setup_llm_judge_configuration(config=llmaj_config)
result




 Waiting for end of deleting monitor definition summarization_quality 




finished

---------------------------------------------------
 Successfully finished deleting monitor definition 
---------------------------------------------------





 Waiting for end of adding monitor definition summarization_quality 




finished

-------------------------------------------------
 Successfully finished adding monitor definition 
-------------------------------------------------




{'custom_monitor_id': 'summarization_quality',
 'llm_provider_id': '019d2997-e859-7924-8192-d12d5b8f9f0d',
 'llm_provider_name': 'LLM_Evaluator_watsonx_00000000-0000-0000-0000-000000000000',
 'llm_provider': 'watsonx.ai',
 'subscription_id': '019d2997-f3ae-797f-9cf2-a3552bb9aeb2',
 'service_provider_id': '019d1b80-d623-7dd1-8ccd-6c958290782d',
 'monitor_instance_id': '019d2998-0800-74ee-93bb-d752849922f7',
 'custom_dataset_id': '019d2998-0b2e-756e-bc6e-923651775217',
 'custom_dataset_table_name': 'aiopenscale18.summarization_quality_019d2997-f3ae-797f-9cf2-a3552bb9aeb2',
 'prompt_setup_response': {'subscription_id': '019d2997-f3ae-797f-9cf2-a3552bb9aeb2',
  'service_provider_id': '019d1b80-d623-7dd1-8ccd-6c958290782d',
  'status': {'state': 'FINISHED'},
  'full_response': {'prompt_template_asset_id': 'b199df20-b88d-4846-989a-1abadd7ff54f',
   'space_id': 'cbcbfcbf-e6ad-4fbf-ad0f-673cd5ba10cc',
   'deployment_id': '72055147-d23c-4b62-95a2-93b7984c5f46',
   'service_provider_id': '019d1b

In [20]:
# Extract the key IDs from the result
monitor_instance_id     = result["monitor_instance_id"]
subscription_id         = result["prompt_setup_response"]["subscription_id"]
mrm_monitor_instance_id = result["prompt_setup_response"]["full_response"]["mrm_monitor_instance_id"]
custom_dataset_id       = result["custom_dataset_id"]  # For record-level metrics

print(f"   Custom Monitor ID:    {result['custom_monitor_id']}")
print(f"   Monitor Instance ID:  {monitor_instance_id}")
print(f"   LLM Provider ID:      {result['llm_provider_id']}")
print(f"   Subscription ID:      {subscription_id}")
print(f"   Custom Dataset ID:    {custom_dataset_id}")

   Custom Monitor ID:    summarization_quality
   Monitor Instance ID:  019d2998-0800-74ee-93bb-d752849922f7
   LLM Provider ID:      019d2997-e859-7924-8192-d12d5b8f9f0d
   Subscription ID:      019d2997-f3ae-797f-9cf2-a3552bb9aeb2
   Custom Dataset ID:    019d2998-0b2e-756e-bc6e-923651775217


## Upload records for production space evaluation

This section is required **only if the user selects a production space**.

`operational_space_id = "production"` : This section must be executed to evaluate production monitoring.

`operational_space_id = "pre_production"` : This section can be skipped.

### Method to get the data_set_id

In [21]:
def get_dataset_id(data_set_type: str):
  data_sets = wos_client.data_sets.list(target_target_id = subscription_id, type = data_set_type).result.data_sets
  data_set_id = None
  if len(data_sets) > 0:
    data_set_id = data_sets[0].metadata.id
  return data_set_id

### Add payload data
Construct payload records from the test data.

In [22]:
import csv
import time

feature_fields = ["input_text"]
prediction = "generated_text"

pl_data = []
prediction_list = []

with open(TEST_DATA_PATH, "r") as csv_file:
    csv_reader = csv.DictReader(csv_file)
    for row in csv_reader:
        request = {"parameters": {"template_variables": {}}}
        for each in feature_fields:
            request["parameters"]["template_variables"][each] = str(row[each])

        predicted_val = row[prediction]
        prediction_list.append(predicted_val)
        response = {"results": [{prediction: predicted_val}]}
        record = {"request": request, "response": response}
        pl_data.append(record)

Store the constructed payload records in OpenScale.

In [23]:
payload_data_set_id = get_dataset_id("payload_logging")

wos_client.data_sets.store_records(
    data_set_id=payload_data_set_id, request_body=pl_data, background_mode=False
)
time.sleep(5)
pl_records_count = wos_client.data_sets.get_records_count(payload_data_set_id)
print("Number of records in the payload logging table: {}".format(pl_records_count))




 Waiting for end of storing records with request id: d0d58729-34ad-4d06-bd88-02497178e642 




active

---------------------------------------
 Successfully finished storing records 
---------------------------------------


Number of records in the payload logging table: 10


### Add feedback data
Construct feedback records from the test data.

In [24]:
test_data_content = []

with open(TEST_DATA_PATH, 'r') as csv_file:
    csv_reader = csv.DictReader(csv_file)
    for row, prediction_val in zip(csv_reader, prediction_list):

        # Read each row from the CSV and add label and prediction values
        result_row = []
        result_row = [row[key] for key in feature_fields if key in row]
        result_row.append(row[label_column])
        result_row.append(prediction_val)
        test_data_content.append(result_row)

if len(test_data_content) == 10: # 10 records are there in the downloaded CSV
    print("generated feedback data from CSV")
else:
    print("Failed to generate feedback data from CSV, Kindly verify the CSV file content")
fields = feature_fields.copy()
fields.append(label_column)
fields.append("_original_prediction")
feedback_data = [{"fields": fields, "values": test_data_content}]

generated feedback data from CSV


Store the feedback records in OpenScale.

In [25]:
feedback_data_set_id = get_dataset_id("feedback")

wos_client.data_sets.store_records(
    data_set_id=feedback_data_set_id, request_body=feedback_data, background_mode=False
)
time.sleep(5)
fb_records_count = wos_client.data_sets.get_records_count(feedback_data_set_id)
time.sleep(10)
print("Number of records in the feedback logging table: {}".format(fb_records_count))




 Waiting for end of storing records with request id: 3d730739-7ada-455d-8201-96b7692ced0e 




active

---------------------------------------
 Successfully finished storing records 
---------------------------------------


Number of records in the feedback logging table: 10


# Risk evaluation

Execute risk evaluation on the RAG model using the test dataset. This process:
- Triggers all configured monitors (including custom LLM-as-a-Judge metrics)
- Computes metric scores for each record and aggregated results
- Returns evaluation status and detailed results


In [26]:
###################################################################################
####### For production deployment space flow
####################################################################################
response  = wos_client.monitor_instances.mrm.evaluate_risk(
    monitor_instance_id=mrm_monitor_instance_id,
    space_id = SPACE_ID,
    background_mode = False
)

eval_result = response.result.to_dict()
print(f"\n✅ Evaluation status: {eval_result['entity']['status']['state']}")
eval_result




 Waiting for risk evaluation of MRM monitor 019d2998-2042-7351-a507-ca3a19607003 




running................
finished

---------------------------------------
 Successfully finished evaluating risk 
---------------------------------------



✅ Evaluation status: finished


{'metadata': {'id': '5fbd4da4-780d-4e2f-8636-f40b49c1e45b',
  'created_at': '2026-03-26T10:03:08.657Z',
  'created_by': 'internal-service'},
 'entity': {'triggered_by': 'user',
  'parameters': {'deployment_id': '72055147-d23c-4b62-95a2-93b7984c5f46',
   'evaluation_start_time': '2026-03-26T10:03:05.719404Z',
   'facts': {'state': 'finished'},
   'measurement_id': '019d2999-0a11-7e11-b251-0ee59537cb3e',
   'monitors_run_status': [{'monitor_id': 'model_health',
     'status': {'state': 'finished'}},
    {'monitor_id': 'summarization_quality', 'status': {'state': 'finished'}}],
   'prompt_template_asset_id': 'b199df20-b88d-4846-989a-1abadd7ff54f',
   'prompt_template_details': {'pta_resource_key': 'facca28d7ea0529b74b6dbf8aeeeb5c05ebcc8e5b636282cc9af7d637bd9ef2a'},
   'space_id': 'cbcbfcbf-e6ad-4fbf-ad0f-673cd5ba10cc',
   'user_iam_id': '1000331001',
   'publish_metrics': 'false',
   'evaluation_tests': []},
  'status': {'state': 'finished',
   'queued_at': '2026-03-26T10:03:08.645000Z',


Uncomment the below code block if using operational_space_id="pre_production".

In [27]:
####################################################################################
######## For pre production deployment space flow
#####################################################################################
# response = wos_client.monitor_instances.mrm.evaluate_risk(
#     monitor_instance_id=mrm_monitor_instance_id,
#     test_data_set_name="summarization_data",
#     test_data_path=TEST_DATA_PATH,
#     content_type="multipart/form-data",
#     body={},
#     space_id=SPACE_ID,
#     includes_model_output=True,   # True because generated_text is already in the CSV
#     background_mode=False         # Wait for completion before returning
# )

# eval_result = response.result.to_dict()
# print(f"\n✅ Evaluation status: {eval_result['entity']['status']['state']}")
# eval_result

# View results

## Display metric scores

The table below shows the **aggregated scores** for each custom metric across all test records.

In [28]:
# Display the computed custom metrics in a table
wos_client.monitor_instances.show_metrics(monitor_instance_id=monitor_instance_id)

2026-03-26 10:05:36.147085+00:00,coherence,019d299b-4a13-70ff-9409-1231b4990006,0.5,0.2,0.8,['computed_on:payload_logging'],summarization_quality,019d2998-0800-74ee-93bb-d752849922f7,f01c28b5-6977-4b6a-ba73-493c497433fa,subscription,019d2997-f3ae-797f-9cf2-a3552bb9aeb2
2026-03-26 10:05:36.147085+00:00,fluency,019d299b-4a13-70ff-9409-1231b4990006,0.95,0.1,0.7,['computed_on:feedback'],summarization_quality,019d2998-0800-74ee-93bb-d752849922f7,f01c28b5-6977-4b6a-ba73-493c497433fa,subscription,019d2997-f3ae-797f-9cf2-a3552bb9aeb2


## Display record-level metrics

Display individual metric scores for each record in the test dataset, showing how each summary was evaluated by the judge LLM.

In [29]:
# Display record-level metrics for each row in the test data
if custom_dataset_id:
    wos_client.data_sets.show_records(data_set_id=custom_dataset_id)
else:
    print("⚠️  Custom dataset ID not available. Record-level metrics cannot be displayed.")

b5c17f98-f7f8-41ab-98cb-27af02ce7e5a,2026-03-26T10:02:32.189Z,None,feedback,2026-03-26T10:05:31.480540Z,f01c28b5-6977-4b6a-ba73-493c497433fa,019d2998-06d9-7df5-969a-548fabe9fe3c,1.0
50ac4690-adda-44b0-a558-4b1c385dfda6,2026-03-26T10:02:32.189Z,None,feedback,2026-03-26T10:05:31.480433Z,f01c28b5-6977-4b6a-ba73-493c497433fa,019d2998-06d9-7df5-969a-548fabe9fe3c,1.0
e5a2aed0-3939-466f-8a0e-a7c65d6316e5,2026-03-26T10:02:32.189Z,None,feedback,2026-03-26T10:05:31.480397Z,f01c28b5-6977-4b6a-ba73-493c497433fa,019d2998-06d9-7df5-969a-548fabe9fe3c,1.0
900bedfc-8dd5-4b19-a268-fea11c24c8db,2026-03-26T10:02:32.189Z,None,feedback,2026-03-26T10:05:31.480362Z,f01c28b5-6977-4b6a-ba73-493c497433fa,019d2998-06d9-7df5-969a-548fabe9fe3c,1.0
2b918b6b-538b-4350-ae5a-db21652e7275,2026-03-26T10:02:32.189Z,None,feedback,2026-03-26T10:05:31.480327Z,f01c28b5-6977-4b6a-ba73-493c497433fa,019d2998-06d9-7df5-969a-548fabe9fe3c,0.5
18b9c1f1-4e13-4b1b-85f7-6f4c776c1244,2026-03-26T10:02:32.189Z,None,feedback,2026-03-26T10:05:31.480291Z,f01c28b5-6977-4b6a-ba73-493c497433fa,019d2998-06d9-7df5-969a-548fabe9fe3c,1.0
567e8af1-8e1b-4ff8-9317-abe14daeff09,2026-03-26T10:02:32.189Z,None,feedback,2026-03-26T10:05:31.480253Z,f01c28b5-6977-4b6a-ba73-493c497433fa,019d2998-06d9-7df5-969a-548fabe9fe3c,1.0
f4de6e69-b44b-45ea-a8e4-728c8afc62b6,2026-03-26T10:02:32.189Z,None,feedback,2026-03-26T10:05:31.480210Z,f01c28b5-6977-4b6a-ba73-493c497433fa,019d2998-06d9-7df5-969a-548fabe9fe3c,1.0
f6c49237-4c7b-4b72-9139-e4585c22b362,2026-03-26T10:02:32.189Z,None,feedback,2026-03-26T10:05:31.480145Z,f01c28b5-6977-4b6a-ba73-493c497433fa,019d2998-06d9-7df5-969a-548fabe9fe3c,1.0
21687a38-b4b3-46bb-b02b-d05793226572,2026-03-26T10:02:32.189Z,None,feedback,2026-03-26T10:05:31.479850Z,f01c28b5-6977-4b6a-ba73-493c497433fa,019d2998-06d9-7df5-969a-548fabe9fe3c,1.0


## View Factsheet information for space subscription

Users can navigate to the project to view the facts published in the factsheet.

In [ ]:
factsheets_url = f"{CPD_URL}/ml-runtime/deployments/{deployment_id}/details?space_id={SPACE_ID}&context=wx&flush=true"

print(f"User can navigate to the published facts in deployment space {factsheets_url}")

## Congratulations!

You have finished the hands-on lab for IBM Watson OpenScale. You can now navigate to the prompt template asset in your deployment space and click on the Evaluate tab to visualise the results on the UI.